# Verify & Benchmark Robot Segmentation (Steps 4–7 + Stabilized Meta SAM 3.1)

Compares real-time students (`SmallSegUNet`, `YOLO26-Sem`) trained on **motion partial labels** vs. **stabilized Meta SAM 3.1 labels**, directly alongside **SAM 3.1 Raw** and **SAM 3.1 Stabilized (Stages A+C)** on:
- Moving & paused validation frames
- PCB leakage & pause stability (Before vs. After stabilization)
- Agreement vs. Stabilized SAM 3.1 (IoU and Boundary F1)
- Embedded H.264 validation comparison video
- Paced 10 Hz latency & freshness benchmark (100 ms tick budget)

In [ ]:
pack_path = "/Users/lboloni/Documents/Develop/Data/BerryPicker-Demopacks/random-both-cameras-video"
result_dir = "/Users/lboloni/Documents/Develop/Data/BerryPicker-Flows/RobotSegmentation_pilot"
camera = "dev2"
input_size = 256
epochs = 12
batch_size = 16
frames_per_episode = 60
train_episodes = None
val_episodes = None
yolo_weights = "yolo26n-seg.yaml"
include_sam3 = True
sam3_checkpoint = "/Users/lboloni/Documents/Develop/Data/BerryPicker-data-external/sam3/sam3.1_multiplex.pt"
sam3_prompt = "robot arm"
sam3_conf = 0.25
sam3_imgsz = 644
retrain = False
retrain_sam3_students = False
benchmark_duration_s = 3.0
review_max_frames = 90
device = "auto"
seed = 0

In [ ]:
import json
import sys
from pathlib import Path

src_root = Path.cwd().resolve()
while src_root.name != "src" and src_root.parent != src_root:
    if (src_root / "_experimental" / "_2026_11_robot_segmentation").is_dir():
        break
    src_root = src_root.parent
if not (src_root / "_experimental" / "_2026_11_robot_segmentation").is_dir():
    src_root = Path("/Users/lboloni/Documents/Develop/Github/BerryPicker/BerryPicker/src")
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

import cv2
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, Markdown, Video, display

from _experimental._2026_11_robot_segmentation.baselines import MedianBackgroundModel, StaticMaskControl
from _experimental._2026_11_robot_segmentation.benchmark import run_paced_benchmark
from _experimental._2026_11_robot_segmentation.data import (
    camera_video_path, detect_pause_flags, frame_motion_scores,
    load_split, make_episode_split, read_video_frames, sample_pose_balanced_indices,
)
from _experimental._2026_11_robot_segmentation.evaluate import (
    boundary_f1, evaluate_predictor, foreground_iou, pause_stability,
)
from _experimental._2026_11_robot_segmentation.models import load_segmenter
from _experimental._2026_11_robot_segmentation.teachers import SAM3DirectSegmenter
from _experimental._2026_11_robot_segmentation.train import load_teacher_samples, train_semantic_student
from _experimental._2026_11_robot_segmentation.viz import (
    make_comparison_strip, plot_latency_comparison, plot_training_curves, write_review_video,
)

out = Path(result_dir)
split = load_split(out / "split.json") if (out / "split.json").is_file() else make_episode_split(pack_path, camera=camera)
tr_eps = list(train_episodes) if train_episodes is not None else split.train
va_eps = list(val_episodes) if val_episodes is not None else split.val

def show_rgb(rgb):
    bgr = cv2.cvtColor(np.asarray(rgb, dtype=np.uint8), cv2.COLOR_RGB2BGR)
    display(Image(data=cv2.imencode(".png", bgr)[1].tobytes()))

bg = MedianBackgroundModel.load(out / "median_background" / camera)
static = StaticMaskControl.load(out / "static_control" / camera)
has_sam3_labels = all((out / "teachers" / "sam3" / camera / ep / "manifest.json").is_file() for ep in tr_eps + va_eps)
has_sam3_ckpt = include_sam3 and Path(sam3_checkpoint).is_file()
consensus_npz = out / "teachers" / f"sam3_static_consensus_{camera}.npz"
pcb_static = np.load(consensus_npz)["pcb_mask"] if consensus_npz.is_file() else None
print("Train eps      :", tr_eps)
print("Val eps        :", va_eps)
print("has_sam3_ckpt  :", has_sam3_ckpt, "| has_sam3_labels:", has_sam3_labels, "| pcb_static_px:", int(pcb_static.sum()) if pcb_static is not None else 0)

## Steps 4–5 — Train (or Load) Students: Motion Teacher vs. Stabilized Meta SAM 3.1

In [ ]:
motion_filter = None
if frames_per_episode is not None:
    motion_filter = {}
    for ep in tr_eps:
        ids, frms = read_video_frames(camera_video_path(pack_path, ep, camera))
        chosen = sample_pose_balanced_indices(frms, count=frames_per_episode, pause_fraction=0.2, seed=seed)
        motion_filter[ep] = [ids[i] for i in chosen]

datasets = {
    "motion": (
        load_teacher_samples(pack_path, out / "teachers", "motion", camera, tr_eps, frame_filter=motion_filter),
        load_teacher_samples(pack_path, out / "teachers", "motion", camera, va_eps),
    )
}
if has_sam3_labels:
    datasets["sam3"] = (
        load_teacher_samples(pack_path, out / "teachers", "sam3", camera, tr_eps),
        load_teacher_samples(pack_path, out / "teachers", "sam3", camera, va_eps),
    )

for tname, ((tr_i, _, _), (va_i, _, _)) in datasets.items():
    print(f"Teacher={tname}: {len(tr_i)} train frames, {len(va_i)} val frames")

jobs = [
    ("UNet (Motion)", "motion", "unet", {"base_channels": 24}),
    ("YOLO26-Sem (Motion)", "motion", "yolo26_semantic", {"weights_or_cfg": yolo_weights, "decoder_channels": 32}),
]
if "sam3" in datasets:
    jobs.extend([
        ("UNet (SAM3.1)", "sam3", "unet", {"base_channels": 24}),
        ("YOLO26-Sem (SAM3.1)", "sam3", "yolo26_semantic", {"weights_or_cfg": yolo_weights, "decoder_channels": 32}),
    ])

summaries = {}
student_dirs = {}
for label_name, tname, arch, kwargs in jobs:
    run_dir = out / "models" / f"{arch}_{tname}_{camera}"
    sum_path = run_dir / "training_summary.json"
    (tr_imgs, tr_lbls, tr_recs), (va_imgs, va_lbls, _) = datasets[tname]
    do_train = retrain or (tname == "sam3" and retrain_sam3_students) or not (run_dir / "best" / "config.json").is_file()
    if do_train:
        summaries[label_name] = train_semantic_student(
            tr_imgs, tr_lbls, va_imgs, va_lbls,
            out_dir=run_dir, arch=arch, arch_kwargs=kwargs,
            camera_id=camera, input_size=input_size, epochs=epochs,
            batch_size=batch_size, seed=seed, device=device,
            split=split.to_dict(), train_records=tr_recs,
        )
    else:
        summaries[label_name] = json.loads(sum_path.read_text(encoding="utf-8"))
    student_dirs[label_name] = run_dir / "best"

fig = plot_training_curves(summaries)
fig.savefig(out / f"training_curves_{camera}.png", dpi=130)
plt.show()

## Side-by-Side Validation Frames, Stabilization Audit & Pause Stability

In [ ]:
predictors = {
    "Median-BG": bg,
    "Static": static,
}
for name, pdir in student_dirs.items():
    predictors[name] = load_segmenter(pdir, device=device)

sam3_raw = None
sam3_stab = None
if has_sam3_ckpt:
    raw_dir = out / "models" / f"sam3_raw_{camera}"
    stab_dir = out / "models" / f"sam3_direct_{camera}"
    sam3_raw = load_segmenter(raw_dir, device=device) if (raw_dir / "config.json").is_file() else SAM3DirectSegmenter(
        checkpoint=sam3_checkpoint, prompt=sam3_prompt, conf=sam3_conf, imgsz=sam3_imgsz, device=device, camera_id=camera, selection_mode="union"
    )
    sam3_stab = load_segmenter(stab_dir, device=device) if (stab_dir / "config.json").is_file() else SAM3DirectSegmenter(
        checkpoint=sam3_checkpoint, prompt=sam3_prompt, conf=sam3_conf, imgsz=sam3_imgsz, device=device, camera_id=camera, selection_mode="union", bg_model=bg, pcb_mask=pcb_static
    )
    predictors["SAM 3.1 Raw"] = sam3_raw
    predictors["SAM 3.1 Stab"] = sam3_stab

val_ep = va_eps[0]
ids, frms = read_video_frames(camera_video_path(pack_path, val_ep, camera))
motion = frame_motion_scores(frms)
pauses, _ = detect_pause_flags(motion)

sub_frms = frms[:review_max_frames]
sub_pauses = pauses[:review_max_frames]
cached_preds = {}
p_rows = ["| Method | Pause runs | Pause frames | Flicker rate | Adj IoU | Max area drop | Static PCB recall % (mean / std) |", "|---|---:|---:|---:|---:|---:|---:|"]
for name, pred in predictors.items():
    if hasattr(pred, "reset_state"):
        pred.reset_state()
    seq = [pred.segment(f) for f in sub_frms]
    cached_preds[name] = np.stack(seq, axis=0)
    st = pause_stability(seq, sub_pauses)
    fl = f"{st['mean_pause_flicker_rate']:.5f}" if st['mean_pause_flicker_rate'] is not None else "n/a"
    ai = f"{st['mean_pause_adj_iou']:.4f}" if st['mean_pause_adj_iou'] is not None else "n/a"
    dr = f"{100*st['max_pause_area_drop_frac']:.1f}%" if st['max_pause_area_drop_frac'] is not None else "n/a"
    if pcb_static is not None and pcb_static.any():
        pcb_recs = []
        for m, f in zip(seq, sub_frms):
            vis_pcb = (pcb_static > 0) & (bg.diff_map(f) < 0.10)
            if vis_pcb.any():
                pcb_recs.append(100.0 * float(((m > 0) & vis_pcb).sum()) / float(vis_pcb.sum()))
        pcb_str = f"{np.mean(pcb_recs):.1f}% / {np.std(pcb_recs):.1f}%" if pcb_recs else "n/a"
    else:
        pcb_str = "n/a"
    p_rows.append(f"| {name} | {st['n_pause_runs']} | {st['pause_frames']} | {fl} | {ai} | {dr} | {pcb_str} |")

panel_order = [n for n in ["Median-BG", "SAM 3.1 Raw", "SAM 3.1 Stab", "UNet (SAM3.1)", "YOLO26-Sem (SAM3.1)"] if n in cached_preds]
move_idx = np.flatnonzero(~sub_pauses)
pause_idx = np.flatnonzero(sub_pauses)
picks = []
if len(move_idx):
    picks.extend(move_idx[np.linspace(0, len(move_idx) - 1, min(3, len(move_idx))).astype(int)].tolist())
if len(pause_idx):
    picks.extend(pause_idx[np.linspace(0, len(pause_idx) - 1, min(2, len(pause_idx))).astype(int)].tolist())
for idx in picks:
    fidx = ids[idx]
    masks = {name: cached_preds[name][idx] for name in panel_order}
    badge = f"f{fidx} {'PAUSE' if sub_pauses[idx] else 'MOVE'}"
    show_rgb(make_comparison_strip(sub_frms[idx], masks, badge=badge))

if has_sam3_labels:
    va_s_imgs, va_s_lbls, _ = datasets["sam3"][1]
    a_rows = ["| Method | Mean IoU vs Stabilized SAM 3.1 | Boundary F1 (2px) vs Stabilized SAM 3.1 |", "|---|---:|---:|"]
    for name, pred in predictors.items():
        if name == "SAM 3.1 Stab":
            continue
        if hasattr(pred, "reset_state"):
            pred.reset_state()
        ious = [foreground_iou(pred.segment(im), (lb == 1)) for im, lb in zip(va_s_imgs, va_s_lbls)]
        bf1s = [boundary_f1(pred.segment(im), (lb == 1), tolerance_px=2) for im, lb in zip(va_s_imgs, va_s_lbls)]
        a_rows.append(f"| {name} | {np.mean(ious):.4f} | {np.mean(bf1s):.4f} |")
    display(Markdown("### Agreement with Stabilized Meta SAM 3.1 on Held-Out Validation Episodes (`" + ", ".join(va_eps) + "`)\n\n" + "\n".join(a_rows)))

display(Markdown(f"### Validation Pause Stability & Static PCB Flicker (`{val_ep}`, first {len(sub_frms)} frames)\n\n" + "\n".join(p_rows)))

In [ ]:
review_mp4 = out / f"review_{val_ep}_{camera}.mp4"
video_masks = {name: cached_preds[name] for name in panel_order}
write_review_video(
    sub_frms,
    predictors={},
    out_mp4=review_mp4,
    fps=10.0,
    extra_masks=video_masks,
    pause_flags=sub_pauses,
)
print("Saved review video:", review_mp4)
display(Video(str(review_mp4), embed=True))

## Paced 10 Hz Latency & Freshness Benchmark (100 ms Tick Budget)

In [ ]:
v_dev2 = camera_video_path(pack_path, val_ep, camera)
v_other = camera_video_path(pack_path, val_ep, "dev3" if camera == "dev2" else "dev2")

bench_reports = {}
bench_1cam_names = [n for n in ["Median-BG", "UNet (SAM3.1)", "UNet (Motion)", "YOLO26-Sem (SAM3.1)", "YOLO26-Sem (Motion)"] if n in predictors]
seen_arch = set()
selected_bench = []
for n in bench_1cam_names:
    key = n.split(" (")[0]
    if key not in seen_arch:
        seen_arch.add(key)
        selected_bench.append(n)

for name in selected_bench:
    slug = name.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_").replace(".", "")
    bench_reports[f"{name} (1 cam)"] = run_paced_benchmark(
        [predictors[name]], [v_dev2],
        duration_s=benchmark_duration_s, target_hz=10.0, warmup_ticks=5,
        out_path=out / f"bench_{slug}_1cam.json",
    )

if v_other.is_file():
    for name in [n for n in selected_bench if n != "Median-BG"]:
        slug = name.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_").replace(".", "")
        bench_reports[f"{name} (2 cams)"] = run_paced_benchmark(
            [predictors[name], predictors[name]], [v_dev2, v_other],
            duration_s=benchmark_duration_s, target_hz=10.0, warmup_ticks=5,
            out_path=out / f"bench_{slug}_2cam.json",
        )

if sam3_stab is not None:
    sam3_stab.reset_state()
    bench_reports["SAM 3.1 Stab (1 cam)"] = run_paced_benchmark(
        [sam3_stab], [v_dev2],
        duration_s=3.0, target_hz=10.0, warmup_ticks=2,
        out_path=out / "bench_sam3_stab_1cam.json",
    )

b_rows = [
    "| Configuration | Cold start (ms) | Median tick (ms) | p95 (ms) | Max (ms) | Max age (ms) | Misses | Drops | 10 Hz Pass |",
    "|---|---:|---:|---:|---:|---:|---:|---:|:---:|",
]
for label, r in bench_reports.items():
    b_rows.append(
        f"| {label} | {r['cold_start_ms']:.1f} | {r['tick_total_ms']['median']:.1f} | "
        f"{r['tick_total_ms']['p95']:.1f} | {r['tick_total_ms']['max']:.1f} | "
        f"{r['frame_age_ms']['max']:.1f} | {r['deadline_misses']} | {r['dropped_frames']} | "
        f"{'PASS' if r['observed_10hz_pass'] else 'FAIL'} |"
    )
display(Markdown("\n".join(b_rows)))

fig = plot_latency_comparison(bench_reports)
fig.savefig(out / f"latency_{camera}.png", dpi=130)
plt.show()

## Independent Reference Evaluation (Runs When Hand-Drawn Validation Masks Are Present)

In [ ]:
eval_manifest = out / "eval_pack" / "eval_manifest.json"
if eval_manifest.is_file():
    mdata = json.loads(eval_manifest.read_text(encoding="utf-8"))
    has_all = bool(mdata["val"]) and all((eval_manifest.parent / it["mask_rel"]).is_file() for it in mdata["val"])
    if has_all:
        e_rows = ["| Method | IoU | Boundary F1 | Base Recall | Gripper Recall | Shadow Leak | Gap Open Rate |", "|---|---:|---:|---:|---:|---:|---:|"]
        for name, pred in predictors.items():
            res = evaluate_predictor(pred, eval_manifest, "val", out / "eval_val" / name, input_size=input_size)
            ov = res["overall"]
            fmt = lambda v: f"{v:.4f}" if v is not None else "n/a"
            e_rows.append(f"| {name} | {fmt(ov['mean_iou'])} | {fmt(ov['mean_boundary_f1'])} | {fmt(ov['mean_base_recall'])} | {fmt(ov['mean_gripper_recall'])} | {fmt(ov['mean_shadow_leakage'])} | {fmt(ov['jaw_gap_open_rate'])} |")
        display(Markdown("\n".join(e_rows)))
    else:
        print("Independent validation reference masks not drawn yet under:", eval_manifest.parent / "val")
else:
    print("Run Explore_RobotSegmentation.ipynb first to export eval_pack.")